<a href="https://colab.research.google.com/github/sanmaykant/sec/blob/main/Notebooks/similarity_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/sec-risk-analyzer"

ARTIFACT_PATH = os.path.join(BASE_PATH, "artifacts/topic_data")
MODEL_PATH = os.path.join(BASE_PATH, "models")

print("Paths ready!")

Paths ready!


In [ ]:
import json
import numpy as np

with open(os.path.join(ARTIFACT_PATH, "enriched_chunks.json"), "r") as f:
    enriched_data = json.load(f)

topic_embeddings = np.load(os.path.join(ARTIFACT_PATH, "topic_embeddings.npy"))

print("Data loaded:", len(enriched_data))

Data loaded: 1649


In [ ]:
print(enriched_data)

[{'chunk': 'Item 1A.\xa0\xa0\xa0\xa0Risk Factors The Company’s business, reputation, results of operations, financial condition and stock price can be affected by a number of factors, whether currently known or unknown, including those described below. When any one or more of these risks materialize from time to time, the Company’s business, reputation, results of operations, financial condition and stock price can be materially and adversely affected.', 'topic': 21, 'year': 2024, 'ticker': 'AAPL'}, {'chunk': 'Because of the following factors, as well as other factors affecting the Company’s results of operations and financial condition, past financial performance should not be considered to be a reliable indicator of future performance, and investors should not use historical trends to anticipate results or trends in future periods. This discussion of risk factors contains forward-looking statements. This section should be read in conjunction with Part II, Item 7, “Management’s Discus

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def max_similarity(topic_id, other_topic_ids):
    if not other_topic_ids:
        return 0

    sims = cosine_similarity(
        topic_embeddings[topic_id].reshape(1, -1),
        topic_embeddings[other_topic_ids]
    )

    return sims.max()

In [ ]:
def get_topics(ticker, year):
    return set([
        item["topic"]
        for item in enriched_data
        if item["ticker"] == ticker
        and item["year"] == year
        and item["topic"] != -1
    ])

In [ ]:
def disappearing_risks(ticker, year1, year2, threshold=1):
    topics_1 = get_topics(ticker, year1)
    topics_2 = get_topics(ticker, year2)

    disappearing = []

    for t1 in topics_1:
        sim = max_similarity(t1, list(topics_2))

        if sim < threshold:
            disappearing.append(t1)

    return disappearing

In [ ]:
print("Disappearing Risks (AAPL 2023 → 2024):")

dr = disappearing_risks("AAPL", 2023, 2024)

print(dr)

Disappearing Risks (AAPL 2023 → 2024):
[1, 8, 15, 25, 27, 30, 33, 44, 47, 49, 50, 52, 62]


In [ ]:
from collections import Counter

def get_topic_counts(ticker, year):
    topics = [
        item["topic"]
        for item in enriched_data
        if item["ticker"] == ticker
        and item["year"] == year
        and item["topic"] != -1
    ]
    return Counter(topics)

In [ ]:
def disappearing_with_drop(ticker, year1, year2, drop_ratio=0.7):
    counts1 = get_topic_counts(ticker, year1)
    counts2 = get_topic_counts(ticker, year2)

    disappearing = []

    for topic in counts1:
        freq1 = counts1[topic]
        freq2 = counts2.get(topic, 0)

        if freq2 < freq1 * drop_ratio:
            disappearing.append((topic, freq1, freq2))

    return disappearing

In [ ]:
print("Disappearing Risks (AAPL 2023 → 2024):")

dr = disappearing_with_drop("AAPL", 2023, 2024)

print(dr)

Disappearing Risks (AAPL 2023 → 2024):
[(52, 2, 1), (54, 3, 2), (25, 6, 3), (33, 3, 2), (0, 3, 2)]


In [ ]:
def risk_coverage_score(ticker, year):
    topics = get_topics(ticker, year)
    return len(topics)

In [ ]:
print("Coverage Scores:")

print("AAPL 2024:", risk_coverage_score("AAPL", 2024))
print("META 2024:", risk_coverage_score("META", 2024))

Coverage Scores:
AAPL 2024: 34
META 2024: 48


In [ ]:
import pandas as pd

topic_info = pd.read_csv(os.path.join(ARTIFACT_PATH, "topic_info.csv"))

In [ ]:
def topic_words(topic_id):
    row = topic_info[topic_info["Topic"] == topic_id]
    if row.empty:
        return "Unknown"
    return row.iloc[0]["Name"]

In [ ]:
def print_disappearing_topics(disappearing_list):
    for topic, freq1, freq2 in disappearing_list:
        name = topic_words(topic)
        drop_pct = ((freq1 - freq2) / freq1) * 100
        print(f"• {name}")
        print(f"  Mentions: {freq1} → {freq2} ({drop_pct:.1f}% decrease)\n")

In [ ]:
print("Disappearing Risks:")
print_disappearing_topics(dr)

Disappearing Risks:
• 52_issues_defects_hardware_software
  Mentions: 2 → 1 (50.0% decrease)

• 54_content_reasonable_commercially_commercially reasonable
  Mentions: 3 → 2 (33.3% decrease)

• 25_legal_companys_contingencies_legal proceedings
  Mentions: 6 → 3 (50.0% decrease)

• 33_foreign_currencies_dollar_foreign currencies
  Mentions: 3 → 2 (33.3% decrease)

• 0_tax_taxes_tax rate_effective tax
  Mentions: 3 → 2 (33.3% decrease)



In [ ]:
print("Disappearing Risks (META 2022 → 2023):")

r = disappearing_with_drop("META", 2022, 2023)

print(r)

Disappearing Risks (META 2022 → 2023):
[(12, 6, 2), (39, 2, 1), (63, 3, 2), (4, 8, 3), (57, 3, 2)]


In [ ]:
print("Disappearing Risks:")
print_disappearing_topics(r)

Disappearing Risks:
• 12_pandemic_covid19_covid19 pandemic_companys
  Mentions: 6 → 2 (66.7% decrease)

• 39_management_financial resources_growth_expansion
  Mentions: 2 → 1 (50.0% decrease)

• 63_lawsuits_class action_class_action
  Mentions: 3 → 2 (33.3% decrease)

• 4_payments_payment_card_money
  Mentions: 8 → 3 (62.5% decrease)

• 57_software hardware_errors_errors bugs_bugs
  Mentions: 3 → 2 (33.3% decrease)



In [ ]:
print("Disappearing Risks (AMZN 2020 → 2021):")

ar = disappearing_with_drop("AMZN", 2020, 2021)

print(r)

Disappearing Risks (AMZN 2020 → 2021):
[(12, 6, 2), (39, 2, 1), (63, 3, 2), (4, 8, 3), (57, 3, 2)]


In [ ]:
print("Disappearing Risks:")
print_disappearing_topics(r)

Disappearing Risks:
• 12_pandemic_covid19_covid19 pandemic_companys
  Mentions: 6 → 2 (66.7% decrease)

• 39_management_financial resources_growth_expansion
  Mentions: 2 → 1 (50.0% decrease)

• 63_lawsuits_class action_class_action
  Mentions: 3 → 2 (33.3% decrease)

• 4_payments_payment_card_money
  Mentions: 8 → 3 (62.5% decrease)

• 57_software hardware_errors_errors bugs_bugs
  Mentions: 3 → 2 (33.3% decrease)

